# 26. Ders: MoodJar'ı Web'e Taşımak (Duygu Analizi Arayüzü)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Bir restoran mutfağını düşünün (Backend / Yapay Zeka Modeli). Orada devasa ocaklar, karmaşık aletler ve usta şefler vardır. Müşteri (Kullanıcı) ise asla mutfağa girmez. O, şık restoranda güzel bir masaya (Gradio / Frontend) oturur, menüden yemeğini seçip garsona söyler. Garson siparişi mutfağa iletir ve pişen yemeği alıp şık bir tabakta müşteriye sunar.

11. derste MoodJar'ın (His Kavanozu) mutfağını ve şefini (BERT duygu analizi modelini) kurmuştuk. Ancak o zaman müşteriyi siyah terminal (while True) ekranına sokmuştuk. Şimdi Gradio garsonunu araya koyarak MoodJar projemizi herkesin telefonundan veya bilgisayarından rahatça girebileceği, emojilerle dolu harika bir web sitesine çevireceğiz.

## Adım 2: Teknik Terimler ve Sözlük

- Integration (Entegrasyon): Yazılan iki farklı sistemin (Yapay Zeka ve Web Arayüzünün) birbirine hata vermeden bağlanması işlemidir.
- Callback (Geri Çağırma): Arayüzdeki butona basıldığında, arka plandaki yapay zeka fonksiyonunun tetiklenip uyanması.
- Textbox (Metin Kutusu): Kullanıcının uzun yazılar girebildiği veya sonuçları okuduğu arayüz bileşeni (Component).

## Adım 3: Kütüphaneleri ve Yapay Zeka Modelini Hazırlama

Gradio'yu ve Hugging Face Transformers kütüphanesini aynı anda yüklüyoruz.

In [ ]:
# Eğer Gradio yüklü değilse !pip install gradio transformers komutunu çalıştırın.
import gradio as gr
from transformers import pipeline
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

# MoodJar'ın kalbi olan Türkçe Duygu Analizi Modelini İndiriyoruz
duygu_modeli = pipeline("sentiment-analysis", model="savasy/bert-base-turkish-sentiment-cased")

print("Mutfağımız (Yapay Zeka Modeli) kuruldu!")

## Adım 4: MoodJar Çekirdek Fonksiyonunu (Backend) Yazma

Daha önce 11. derste yazdığımız fonksiyonu biraz güncelliyoruz. Gradio arayüzü `return` ile dönen şeyi aynen ekrana basacağı için, cevabımızı şık, emojili bir string (metin) olarak şekillendireceğiz.

In [ ]:
import random

def moodjar_analiz_et(kullanici_metni):
    # Eğer kullanıcı hiçbir şey yazmazsa hata vermemesi için koruma (Fallback)
    if len(kullanici_metni.strip()) < 3:
        return "Lütfen kavanoza anlamlı ve biraz daha uzun bir his bırak."
        
    # Metni yapay zekaya (modele) veriyoruz
    analiz = duygu_modeli(kullanici_metni)[0]
    etiket = analiz['label']
    skor = analiz['score']
    
    # Emojili ve şık bir çıktı tasarımı
    if etiket == "positive":
        yanit = random.choice([
            "Harika hissediyorsun! Bu güzel enerjinin tadını çıkar. ☀️",
            "Ne güzel bir haber. Kavanozumuz bu mutlulukla parlıyor! ✨"
        ])
        durum = f"Duygu: Pozitif (Mutlu) 😊 (Yapay Zeka Eminlik Oranı: %{skor*100:.1f})"
        
    else:
        yanit = random.choice([
            "Görünüşe göre zor bir anındasın. Derin bir nefes al, her şey geçecek. 🌧️",
            "Anlıyorum. Bazen işler ters gidebilir, önemli olan pes etmemek. ☂️"
        ])
        durum = f"Duygu: Negatif (Üzgün) 😔 (Yapay Zeka Eminlik Oranı: %{skor*100:.1f})"
        
    # Çıktıları iki satır halinde (araya \n koyarak) birleştirip geri döndürüyoruz.
    return f"{durum}\n\nMoodJar'ın Sana Mesajı: {yanit}"

## Adım 5: Gradio Arayüzü İçinde Modeli Çağırma ve Tasarım

Şimdi bu fonksiyonu web sitesine bağlıyoruz. Kullanıcının uzun bir cümle yazması için girdi kutusunu (Textbox) biraz büyüteceğiz.

In [ ]:
# lines=3 diyerek metin kutusunun 3 satır büyüklüğünde olmasını sağlıyoruz
girdi_kutusu = gr.Textbox(lines=3, placeholder="Bugün kendini nasıl hissediyorsun? Buraya yaz...", label="Düşüncelerini Kavanoza Bırak")

# Çıktı kutusunu da özelleştirelim
cikti_kutusu = gr.Textbox(label="Yapay Zeka Analiz Sonucu")

moodjar_arayuzu = gr.Interface(
    fn=moodjar_analiz_et,
    inputs=girdi_kutusu,
    outputs=cikti_kutusu,
    title="🧠 MoodJar - His Kavanozu",
    description="Bu yapay zeka destekli kavanoz, yazdığın cümlelerin duygusunu analiz eder ve seninle empati kurar."
)

print("Tasarım tamamlandı, uygulama yayına hazırlanıyor...")

## Adım 6: Uygulamayı Yayınlama (Launch)

Uygulamamızı başlatalım ve linki alalım.

In [ ]:
moodjar_arayuzu.launch(share=True)

## Adım 7: Dene ve Keşfet - Gerçek İnsan Testi

Yukarıda kod bloğunda çıkan public URL'yi (https://....gradio.live) hemen telefonunuzun tarayıcısına yazın (Veya arkadaşınıza gönderin). Telefonunuzdan uzun bir paragraf ("Bugün çok garip bir gündü, önce yağmur yağdı canım sıkıldı, sonra güneşi görünce mutlu oldum" gibi) yazarak modelin (Kavanozun) size nasıl tepki verdiğini ve ekranda nasıl şık durduğunu inceleyin.

## Adım 8: Sıra Sende Görevi - Toksik Mesaj Güvenlik Duvarı

Problem: MoodJar uygulamanız çok popüler oldu, ama bazı kullanıcılar kavanoza küfür veya nefret dolu şeyler yazmaya başladı. GCA Coding Jam'in 17. Dersinde yazdığımız Watchdog (Veri Bekçisi) altyapısını bu arayüze (Frontend) entegre etmeniz isteniyor.

Gereksinimler:
1. Önce toksisite modelini indirin: `bekci = pipeline("text-classification", model="unitary/toxic-bert")`
2. `moodjar_analiz_et` fonksiyonunu kopyalayarak `guvenli_moodjar(metin)` adında yeni bir fonksiyon yazın.
3. Fonksiyonun en başında (kullanıcı metni daha duygu analizine GİRMEDEN ÖNCE), metni `bekci` modeline verin.
4. Eğer toksisite skoru 0.60'tan büyükse; duygu analizi yapmayı YARIDA KESİN ve doğrudan `return "SİSTEM UYARISI: Lütfen kavanoza kötü kelimeler atmayın!"` şeklinde hata mesajı döndürün.
5. Eğer toksik değilse, kod normal işleyişine (Duygu analizi ve empati cevapları) devam etsin.
6. Yeni fonksiyonunuzla yeni bir `gr.Interface` kurup test edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 26. Dersi tamamlayarak ilk projeniz olan MoodJar'ı başarıyla buluta / web ortamına taşıdınız. Artık o sadece bir kod değil, insanların kullanabileceği yaşayan bir dijital ürün.

Bir sonraki dersimizde, Yılın Şiiri (Metin Üretimi) projesini web arayüzüne taşıyacak ve kullanıcılara (Slider ile) yapay zekanın yaratıcılık seviyesini seçme imkanı sunacağız.